
# Business Insights and Visualization

This notebook turns the distributed pipeline outputs into decisions.

## Business problem

The company has event and customer profile data but lacks a repeatable way to:

- identify high-value customer groups,
- find low-engagement or high-churn populations,
- measure regional support burden,
- understand which event paths convert to purchases,
- and reuse the platform for other datasets.

The notebook can read either local sample files or the same CSVs from Amazon S3.


In [ ]:

from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

ROOT=Path("..").resolve()
sys.path.append(str(ROOT))
from scripts.orchestration.s3_loader import load_dataset

profiles=load_dataset(
    ROOT/"data/sample/customer_profiles.csv",
    "data/raw/customer_behavior/customer_profiles.csv"
)
events=load_dataset(
    ROOT/"data/sample/customer_events.csv",
    "data/raw/customer_behavior/customer_events.csv",
    parse_dates=["event_ts"]
)
profiles.shape, events.shape


## 1. Executive KPIs

In [ ]:

purchase_events=events[events["event_type"]=="purchase"].copy()
kpis=pd.Series({
    "customers":len(profiles),
    "events":len(events),
    "purchase_revenue":purchase_events["amount"].sum(),
    "purchase_conversion_rate":(events["event_type"]=="purchase").mean(),
    "customer_churn_rate":profiles["churned"].mean(),
    "average_engagement":profiles["engagement_score"].mean(),
})
display(kpis.to_frame("value").round(3))



## 2. Revenue concentration by customer segment

This shows where revenue is concentrated and where differentiated service or retention investments may create the most value.


In [ ]:

revenue_by_customer=purchase_events.groupby("customer_id",as_index=False)["amount"].sum().rename(columns={"amount":"revenue"})
analysis=profiles.merge(revenue_by_customer,on="customer_id",how="left").fillna({"revenue":0})
seg=analysis.groupby("seed_segment").agg(
    customers=("customer_id","count"),
    revenue=("revenue","sum"),
    avg_engagement=("engagement_score","mean"),
    churn_rate=("churned","mean"),
    avg_support_calls=("support_calls_90d","mean"),
).sort_values("revenue",ascending=False)
display(seg.round(2))

fig, ax=plt.subplots(figsize=(10,5.5))
bars=ax.bar(seg.index,seg["revenue"])
ax.set_title("Revenue by customer segment")
ax.set_xlabel("Customer segment")
ax.set_ylabel("Revenue")
ax.grid(axis="y",alpha=.25)
for b,v in zip(bars,seg["revenue"]):
    ax.text(b.get_x()+b.get_width()/2,v,f"${v:,.0f}",ha="center",va="bottom",fontsize=9)
plt.tight_layout()
plt.show()



## 3. Churn and engagement risk

The relationship between engagement, support burden, and churn helps prioritize customer-success interventions.


In [ ]:

risk=profiles.groupby("seed_segment").agg(
    churn_rate=("churned","mean"),
    avg_engagement=("engagement_score","mean"),
    avg_support_calls=("support_calls_90d","mean"),
    avg_returns=("returns_90d","mean")
).sort_values("churn_rate",ascending=False)
display(risk.round(3))

fig, ax=plt.subplots(figsize=(9,5.5))
for name,group in profiles.groupby("seed_segment"):
    ax.scatter(group["engagement_score"],group["support_calls_90d"],s=18,alpha=.45,label=name)
ax.set_title("Engagement versus support burden")
ax.set_xlabel("Engagement score")
ax.set_ylabel("Support calls in 90 days")
ax.grid(alpha=.2)
ax.legend(ncol=2,loc="upper center",bbox_to_anchor=(.5,-.12))
plt.tight_layout()
plt.show()



## 4. Regional operating burden

High event volume is good only when support and return rates remain manageable.


In [ ]:

regional=profiles.groupby("region").agg(
    customers=("customer_id","count"),
    churn_rate=("churned","mean"),
    avg_support_calls=("support_calls_90d","mean"),
    avg_returns=("returns_90d","mean"),
    avg_engagement=("engagement_score","mean")
)
regional=regional.join(purchase_events.groupby("region")["amount"].sum().rename("revenue"))
display(regional.round(3))

fig, ax=plt.subplots(figsize=(10,5.5))
x=np.arange(len(regional))
w=.35
ax.bar(x-w/2,regional["avg_support_calls"],w,label="Support calls")
ax.bar(x+w/2,regional["avg_returns"],w,label="Returns")
ax.set_xticks(x,regional.index)
ax.set_title("Regional service burden")
ax.set_ylabel("Average count per customer")
ax.grid(axis="y",alpha=.25)
ax.legend()
plt.tight_layout()
plt.show()



## 5. Event funnel

The event funnel shows how much customer activity progresses from browsing to purchase.


In [ ]:

order=["view","search","cart","purchase","support","return"]
funnel=events["event_type"].value_counts().reindex(order).fillna(0).astype(int)
display(funnel.to_frame("events"))

fig, ax=plt.subplots(figsize=(10,5.5))
bars=ax.bar(funnel.index,funnel.values)
ax.set_title("Customer behavior event funnel")
ax.set_xlabel("Event type")
ax.set_ylabel("Event count")
ax.grid(axis="y",alpha=.25)
for b,v in zip(bars,funnel.values):
    ax.text(b.get_x()+b.get_width()/2,v,f"{v:,}",ha="center",va="bottom",fontsize=9)
plt.tight_layout()
plt.show()



## 6. Reusable K-Means customer segmentation

The cluster model uses business behavior rather than arbitrary identifiers. Cluster profiles are the real output; the cluster number itself has no inherent meaning.


In [ ]:

features=[
    "monthly_income","tenure_months","digital_logins_30d",
    "purchase_frequency_30d","avg_order_value",
    "support_calls_90d","returns_90d","engagement_score"
]
X=StandardScaler().fit_transform(profiles[features])

scores={}
for k in range(2,7):
    labels=KMeans(n_clusters=k,n_init=15,random_state=42).fit_predict(X)
    scores[k]=silhouette_score(X,labels)

best_k=max(scores,key=scores.get)
model=KMeans(n_clusters=best_k,n_init=20,random_state=42)
profiles["cluster"]=model.fit_predict(X)

print("Selected k:",best_k)
display(pd.Series(scores,name="silhouette_score").round(3))
cluster_profile=profiles.groupby("cluster")[features+["churned"]].mean().round(2)
display(cluster_profile)


In [ ]:

fig, ax=plt.subplots(figsize=(8,5))
ks=list(scores)
vals=list(scores.values())
bars=ax.bar(ks,vals)
ax.set_title("K-Means model selection")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Silhouette score")
ax.set_xticks(ks)
ax.grid(axis="y",alpha=.25)
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2,v,f"{v:.3f}",ha="center",va="bottom",fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:

pca=PCA(n_components=2,random_state=42)
Xp=pca.fit_transform(X)
fig, ax=plt.subplots(figsize=(9,6))
for cluster in sorted(profiles["cluster"].unique()):
    mask=profiles["cluster"].to_numpy()==cluster
    ax.scatter(Xp[mask,0],Xp[mask,1],s=18,alpha=.55,label=f"Cluster {cluster}")
ax.set_title("Customer clusters projected with PCA")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.grid(alpha=.2)
ax.legend(ncol=2,loc="upper center",bbox_to_anchor=(.5,-.12))
plt.tight_layout()
plt.show()



## 7. How to reuse this notebook

Replace the two source files and update `config/dataset_contract.yaml`.

Then change:

- the business KPIs,
- feature names,
- event taxonomy,
- grouping dimensions,
- and cluster profiling logic.

The infrastructure, S3 loader, EMR orchestration, validation approach, and notebook execution pattern remain reusable.
